# Land Cover Classification CNN
## GEOG 6160 Final Project
## Magnus Tveit

Trains a CNN to classify 8×8 Landsat chips into 4 land cover classes:
**Lake**, **River**, **Sediment Bank**, **Rock**.

**Prerequisites:** Run `LandCoverExtractChips.ipynb` first to populate `Chips/`.

Pipeline mirrors `DeltaCNN.ipynb` but is scaled up for multi-class output:
1. Load `.npy` chips from `Chips/train`, `valid`, `test`
2. Visualise a sample of training chips
3. Build and train the CNN
4. Plot training curves
5. Evaluate on the test set
6. Save best model as `landcover_model.keras` → used by `LandCoverPredict.ipynb`

### Set Up

### Settings

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import precision_score, recall_score, confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)

base_dir  = os.getcwd()
chips_dir = os.path.join(base_dir, "Chips")

# Read chip settings from ExtractChips
config_path = os.path.join(base_dir, "experiment_config.json")
with open(config_path) as f:
    config = json.load(f)

CHIP_SIZE          = config["CHIP_SIZE"]
COVERAGE_THRESHOLD = config["COVERAGE_THRESHOLD"]

# CNN-only knob
USE_AUGMENTATION = True   # << change this here

IMAGE_HEIGHT = CHIP_SIZE
IMAGE_WIDTH  = CHIP_SIZE
NUM_CHANNELS = 7

CLASSES     = ["Lake", "River", "Sediment Bank", "Rock"]
NUM_CLASSES = len(CLASSES)
BATCH_SIZE  = 32
EPOCHS      = 50

print(f"CHIP_SIZE          : {CHIP_SIZE}  (from experiment_config.json)")
print(f"COVERAGE_THRESHOLD : {COVERAGE_THRESHOLD}  (from experiment_config.json)")
print(f"USE_AUGMENTATION   : {USE_AUGMENTATION}")
print(f"Input shape        : ({IMAGE_HEIGHT}, {IMAGE_WIDTH}, {NUM_CHANNELS})")
print(f"Classes            : {CLASSES}")

### Load chips

In [ ]:
def load_chips(split):
    """Load all chips for a given split and return (X, Y) arrays."""
    X_list, Y_list = [], []
    for class_idx, cls in enumerate(CLASSES):
        cls_dir = os.path.join(chips_dir, split, cls)
        files   = sorted([f for f in os.listdir(cls_dir) if f.endswith(".npy")])
        print(f"  {split}/{cls}: {len(files)} chips")
        for fname in files:
            chip = np.load(os.path.join(cls_dir, fname)).astype("float32")
            X_list.append(chip)
            # One-hot encode the class label
            label = np.zeros(NUM_CLASSES, dtype="float32")
            label[class_idx] = 1.0
            Y_list.append(label)
    return np.stack(X_list), np.stack(Y_list)

X_train, y_train = load_chips("train")
X_val,   y_val   = load_chips("valid")
X_test,  y_test  = load_chips("test")
print(f"\nX_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

### Visualise chips
Displays a random sample of training chips as NIR/Red/Green false colour composites to sanity-check the extracted data before training.

In [ ]:
def show_chips(X, Y, n=10, title=""):
    fig, axes = plt.subplots(1, n, figsize=(2*n, 2.5))
    indices = np.random.choice(len(X), n, replace=False)
    for ax, idx in zip(axes, indices):
        # Band indices: NIR=3, Red=2, Green=1
        rgb = X[idx][:,:,[3,2,1]]
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
        ax.imshow(rgb)
        ax.set_title(CLASSES[np.argmax(Y[idx])], fontsize=7)
        ax.axis("off")
    plt.suptitle(title, fontweight="bold")
    plt.tight_layout(); plt.show()

show_chips(X_train, y_train, title="Sample training chips — Land Cover")

### Build CNN
Same architecture as `DeltaCNN` but with wider dense layers (256 → 128) to handle
the increased complexity of 4-class discrimination.

In [ ]:
aug_layers = []
if USE_AUGMENTATION:
    aug_layers = [
        layers.RandomFlip("horizontal_and_vertical",
                          input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)),
        layers.RandomRotation(0.25),
    ]
    print("Augmentation: RandomFlip + RandomRotation(0.25) — active during training only")
else:
    aug_layers = [
        layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)),
    ]
    print("Augmentation: OFF")

model = models.Sequential(
    aug_layers +
    [
        # Block 1
        layers.Conv2D(32, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D((2,2)),
        layers.BatchNormalization(),

        # Block 2
        layers.Conv2D(64, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D((2,2)),
        layers.BatchNormalization(),

        # Block 3
        layers.Conv2D(128, (3,3), padding="same", activation="relu"),
        layers.MaxPooling2D((2,2)),
        layers.BatchNormalization(),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ]
)
model.summary()

### Compile

In [ ]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss      = tf.keras.losses.CategoricalCrossentropy(),
    metrics   = ["accuracy"]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath       = os.path.join(base_dir, "landcover_model.keras"),
        save_best_only = True,   # only overwrite if val_loss improves
        monitor        = "val_loss",
        verbose        = 1
    )
]

## 7. Train

In [ ]:
history = model.fit(
    x=X_train, y=y_train,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    shuffle=True,
    callbacks=callbacks
)

## 8. Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(history.history["loss"],     label="Training loss")
ax1.plot(history.history["val_loss"], label="Validation loss")
ax1.set_xlabel("Epochs"); ax1.set_ylabel("Loss")
ax1.set_title("Loss over training epochs"); ax1.legend()
ax2.plot(history.history["accuracy"],     label="Training accuracy")
ax2.plot(history.history["val_accuracy"], label="Validation accuracy")
ax2.set_xlabel("Epochs"); ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy over training epochs"); ax2.legend()
plt.tight_layout(); plt.show()

### Evaluate
Evaluate using the test set

In [ ]:
best_model = keras.models.load_model(os.path.join(base_dir, "landcover_model.keras"))
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

### Precision, recall and confusion matrix

In [ ]:
y_pred_probs = best_model.predict(X_test)
y_pred   = np.argmax(y_pred_probs, axis=1)
y_actual = np.argmax(y_test,       axis=1)

print(f"Precision : {precision_score(y_actual, y_pred, average='macro'):.4f}")
print(f"Recall    : {recall_score(   y_actual, y_pred, average='macro'):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, norm, title in zip(axes, [None, "true"],
                            ["Confusion matrix (counts)",
                             "Confusion matrix (normalised)"]):
    cm  = confusion_matrix(y_actual, y_pred, normalize=norm)
    fmt = "d" if norm is None else ".2f"
    sns.heatmap(cm, annot=True, fmt=fmt, cmap="RdYlGn",
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("Actual",    fontweight="bold")
    ax.set_title(title)
plt.tight_layout(); plt.show()

### Visualise test chip predictions

In [ ]:
n_show  = 15
indices = np.random.choice(len(X_test), n_show, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()
for ax, idx in zip(axes, indices):
    rgb = X_test[idx][:,:,[3,2,1]]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    ax.imshow(rgb)
    actual_lbl    = CLASSES[y_actual[idx]]
    predicted_lbl = CLASSES[y_pred[idx]]
    colour = "green" if actual_lbl == predicted_lbl else "red"
    ax.set_title(f"A: {actual_lbl}\nP: {predicted_lbl}", fontsize=7, color=colour)
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor(colour); spine.set_linewidth(3)
plt.suptitle("Test predictions (green=correct, red=incorrect)", fontweight="bold")
plt.tight_layout(); plt.show()

print("\nDone. Ready for LandCoverPredict.ipynb")

Experiment Log

In [ ]:
import json, os

# ── Only these two need to be set manually — everything else is pulled automatically ──
log_chip_size  = CHIP_SIZE          # pulls from settings cell automatically
log_coverage   = COVERAGE_THRESHOLD # pulls from settings cell automatically  
log_aug        = USE_AUGMENTATION   # pulls from settings cell automatically
# ─────────────────────────────────────────────────────────────────────────────────────

# Pull chip counts from the Chips/train folders
log_counts = {}
for cls in CLASSES:
    folder = os.path.join(chips_dir, "train", cls)
    log_counts[cls] = len([f for f in os.listdir(folder) if f.endswith(".npy")])

# Pull training results from history and test evaluation
log_best_val_acc  = max(history.history["val_accuracy"])
log_best_val_loss = min(history.history["val_loss"])
log_best_epoch    = history.history["val_accuracy"].index(log_best_val_acc) + 1

# test_loss and test_acc come from the evaluate cell — make sure that cell ran first
# (they are set by: test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0))

# Build the log entry
entry = {
    "chip_size"     : log_chip_size,
    "coverage"      : log_coverage,
    "augmentation"  : log_aug,
    "train_chips"   : log_counts,
    "best_val_acc"  : round(log_best_val_acc, 4),
    "best_val_loss" : round(log_best_val_loss, 6),
    "best_epoch"    : log_best_epoch,
    "test_acc"      : round(test_acc, 4),
    "test_loss"     : round(test_loss, 6),
}

# Append to a JSON log file that persists across runs
log_path = os.path.join(base_dir, "experiment_log.json")
if os.path.exists(log_path):
    with open(log_path) as f:
        log = json.load(f)
else:
    log = []

log.append(entry)
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

# Print the full log as a table
print(f"{'#':>2}  {'chip':>4}  {'cov':>4}  {'aug':>5}  {'lake':>5}  {'sed':>5}  {'rock':>5}  {'val_acc':>8}  {'ep':>3}  {'test_acc':>9}  {'test_loss':>10}")
print("-" * 85)
for i, r in enumerate(log):
    c = r["train_chips"]
    print(f"{i+1:>2}  {r['chip_size']:>4}  {r['coverage']:>4}  {str(r['augmentation']):>5}  "
          f"{c.get('Lake',0):>5}  {c.get('Sediment Bank',0):>5}  {c.get('Rock',0):>5}  "
          f"{r['best_val_acc']:>8.4f}  {r['best_epoch']:>3}  {r['test_acc']:>9.4f}  {r['test_loss']:>10.6f}")